# AdaRetrieve routing follow-up
Choose a Tesla T4. Upload AdaRetrieve-project.zip and AdaRetrieve-indexes-and-scores.zip to /content, then run the code cells. The notebook verifies the inputs and writes fresh observations.

In [ ]:
from pathlib import Path
import hashlib, importlib.util, os, subprocess, sys, tempfile, zipfile
import torch
from google.colab import files

assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0)
work = Path(tempfile.mkdtemp(prefix='adaretrieve-', dir='/content'))
def digest(data):
    return hashlib.sha256(data).hexdigest()
with zipfile.ZipFile('/content/AdaRetrieve-project.zip') as z:
    manifest = dict(line.split('  ', 1)[::-1] for line in z.read('SHA256SUMS.txt').decode().splitlines())
    assert set(z.namelist()) == set(manifest) | {'SHA256SUMS.txt'}
    for name, expected in manifest.items():
        assert digest(z.read(name)) == expected, name
        if name.startswith('project/') or name in ('routing-followup/experiment.py', 'routing-followup/protocol.md'):
            z.extract(name, work)
    assert manifest['routing-followup/experiment.py'] == '24bf6e12f0506ed9d09dfafe45b586da9d3c2ef89f08bc55714f04b7f09779bf'
    assert manifest['routing-followup/protocol.md'] == '91b152e15daba90ec8aaafb741318261672b6ed6f5973a080b230ec8e7d6a6d1'
cache = Path('/content/AdaRetrieve-indexes-and-scores.zip')
assert digest(cache.read_bytes()) == '3af2b67a10413a7b6e7de78aea782d1335cb999af380c8b0bb00f89a20176340'
with zipfile.ZipFile(cache) as z:
    z.extractall(work / 'project/cache')
if importlib.util.find_spec('faiss') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'], check=True)
os.chdir(work)
print('Verified inputs:', work, torch.cuda.get_device_name(0))


In [ ]:
subprocess.run([sys.executable,'-u','routing-followup/experiment.py'],check=True)
with zipfile.ZipFile('routing-followup-results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(Path('routing-followup/output').glob('*')):
        if p.is_file(): z.write(p,p.name)
print('RESULTS_READY',Path('routing-followup-results.zip').stat().st_size,flush=True)
files.download('routing-followup-results.zip')